In [1]:
import pandas as pd
import numpy as np
from pyprojroot import here
DATA_RAW= here("data/raw")
DATA_PROC = here("data/processed")

In [2]:
#https://www.groupe-sncf.com/en/commitments/open-data
sncf_national_raw = DATA_RAW / "sncf_national_trainStation.csv"

In [3]:
sncf = pd.read_csv(sncf_national_raw, sep=";")

In [4]:
sncf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2769 entries, 0 to 2768
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Nom                    2769 non-null   object
 1   Trigramme              2769 non-null   object
 2   Segment(s) DRG         2769 non-null   object
 3   Position géographique  2762 non-null   object
 4   Code commune           2769 non-null   int64 
 5   Code(s) UIC            2769 non-null   object
dtypes: int64(1), object(5)
memory usage: 129.9+ KB


In [5]:
sncf.head()

,Nom,Trigramme,Segment(s) DRG,Position géographique,Code commune,Code(s) UIC
0,Abancourt,ABT,C,"49.6852237, 1.7743058",60001,87313759
1,Abbaretz,AAR,C,"47.5546432, -1.5244159",44001,87481614
2,Abbeville,ABB,B,"50.10221, 1.82449",80001,87317362
3,Ablon-sur-Seine,ABL,B,"48.725468, 2.419151",94001,87545269
4,Achères Grand Cormier,GCR,B,"48.9551835, 2.0919031",78551,87386052


In [6]:
# Parse coordinates
latlon = sncf["Position géographique"].str.split(",", n=1, expand=True)
sncf["lat"] = pd.to_numeric(latlon[0].str.strip(), errors="coerce")
sncf["lon"] = pd.to_numeric(latlon[1].str.strip(), errors="coerce")

# Basic integrity flags
sncf["has_geo"] = sncf["lat"].notna() & sncf["lon"].notna()

# Keep key columns for the project
stations_fr = sncf.rename(columns={
    "Code(s) UIC": "station_id",
    "Nom": "name",
    "Trigramme": "trigram",
    "Segment(s) DRG": "segment_drg",
    "Code commune": "code_commune"
})[
    [
        "station_id",
        "name",
        "trigram",
        "segment_drg",
        "code_commune",
        "lat",
        "lon",
        "has_geo"
    ]
]

# Sanity: station_id should be unique and non-null
assert stations_fr["station_id"].is_unique
assert stations_fr["station_id"].notna().all()

stations_fr.to_csv(DATA_PROC / "sncf_national_trainStation-p.csv", index=False)


In [7]:
sncf.head()

,Nom,Trigramme,Segment(s) DRG,Position géographique,Code commune,Code(s) UIC,lat,lon,has_geo
0,Abancourt,ABT,C,"49.6852237, 1.7743058",60001,87313759,49.685224,1.774306,True
1,Abbaretz,AAR,C,"47.5546432, -1.5244159",44001,87481614,47.554643,-1.524416,True
2,Abbeville,ABB,B,"50.10221, 1.82449",80001,87317362,50.102210,1.824490,True
3,Ablon-sur-Seine,ABL,B,"48.725468, 2.419151",94001,87545269,48.725468,2.419151,True
4,Achères Grand Cormier,GCR,B,"48.9551835, 2.0919031",78551,87386052,48.955183,2.091903,True


In [8]:
sncf['has_geo'].value_counts()

has_geo
True     2762
False       7
Name: count, dtype: int64